In [ ]:
# ============================================================
# FEATURE SPACE VISUALIZATION
# Classical vs Hybrid Quantum–Classical
# ----- 
# ============================================================


# 
# -- before running this code  ---   be sure to upload the required data as mentioned below in code
#  --- take care for the path of the file and data that is uploaded for smooth run
# 
import pickle
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
from tensorflow.keras.preprocessing.sequence import pad_sequences

# ============================================================
# CONFIG
# ============================================================
MODEL_PATH = "/kaggle/input/gaurav-paper-results/model/sentiment_model.keras"
TOKENIZER_PATH = "/kaggle/input/classical-tokenizer/tokenizer.pkl"
DATASET_PATH = "/kaggle/input/sentiment140/training.1600000.processed.noemoticon.csv"

MAX_LEN = 50
N_VIS = 3000        # SAFE for t-SNE
BATCH_SIZE = 256
RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)
tf.keras.utils.set_random_seed(RANDOM_STATE)

# ============================================================
# Dummy Quantum Layer (for model loading only)
# ============================================================
class QuantumCF2Layer(tf.keras.layers.Layer):
    def __init__(self, n_qubits, **kwargs):
        super().__init__(**kwargs)
        self.n_qubits = n_qubits

    def call(self, inputs):
        return inputs

    def get_config(self):
        config = super().get_config()
        config.update({"n_qubits": self.n_qubits})
        return config

# ============================================================
# Load Model
# ============================================================
model = tf.keras.models.load_model(
    MODEL_PATH,
    custom_objects={"QuantumCF2Layer": QuantumCF2Layer},
    compile=False
)

print("✔ Model loaded successfully\n")

print("Model layers:")
for layer in model.layers:
    print(" -", layer.name)

# ============================================================
# Find Required Layers Robustly
# ============================================================
def find_layer(model, keyword):
    for layer in model.layers:
        if keyword in layer.name:
            return layer
    raise ValueError(f"Layer containing '{keyword}' not found")

classical_layer = find_layer(model, "global_average_pooling1d")
hybrid_layer    = find_layer(model, "concatenate")

# Feature extractors
classical_extractor = tf.keras.Model(
    inputs=model.input,
    outputs=classical_layer.output
)

hybrid_extractor = tf.keras.Model(
    inputs=model.input,
    outputs=hybrid_layer.output
)

# ============================================================
# Load Tokenizer
# ============================================================
with open(TOKENIZER_PATH, "rb") as f:
    tokenizer = pickle.load(f)

def encode(texts):
    return pad_sequences(
        tokenizer.texts_to_sequences(texts),
        maxlen=MAX_LEN,
        padding="post",
        truncating="post"
    )

# ============================================================
# Load & Sample Dataset (NO TRAINING)
# ============================================================
import pandas as pd

cols = ["target", "id", "date", "query", "user", "text"]
df = pd.read_csv(DATASET_PATH, encoding="latin-1", header=None, names=cols)
df = df[["target", "text"]]
df["target"] = df["target"].replace({4: 1})

df = (
    df.groupby("target", group_keys=False)
      .apply(lambda x: x.sample(N_VIS // 2, random_state=RANDOM_STATE))
      .sample(frac=1.0, random_state=RANDOM_STATE)
      .reset_index(drop=True)
)

X_vis = encode(df.text.values)
y_vis = df.target.values

print(f"\n✔ Using {len(X_vis)} samples for visualization")

# ============================================================
# Extract Feature Representations
# ============================================================
Z_classical = classical_extractor.predict(
    X_vis, batch_size=BATCH_SIZE, verbose=1
)

Z_hybrid = hybrid_extractor.predict(
    X_vis, batch_size=BATCH_SIZE, verbose=1
)

print("\n✔ Feature extraction completed")

# ============================================================
# Quantitative Separability (Silhouette Score)
# ============================================================
sil_classical = silhouette_score(Z_classical, y_vis)
sil_hybrid = silhouette_score(Z_hybrid, y_vis)

print("\n=== FEATURE SEPARABILITY (Silhouette Score) ===")
print(f"Classical BiLSTM : {sil_classical:.4f}")
print(f"Hybrid Quantum   : {sil_hybrid:.4f}")

# ============================================================
# t-SNE Projection
# ============================================================
tsne = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate="auto",
    init="random",
    random_state=RANDOM_STATE
)

Zc_2d = tsne.fit_transform(Z_classical)
Zh_2d = tsne.fit_transform(Z_hybrid)

# ============================================================
# Visualization
# ============================================================
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.scatter(
    Zc_2d[:,0], Zc_2d[:,1],
    c=y_vis, s=6, cmap="coolwarm", alpha=0.6
)
plt.title("Classical BiLSTM Feature Space (t-SNE)")
plt.xlabel("Dim-1")
plt.ylabel("Dim-2")

plt.subplot(1,2,2)
plt.scatter(
    Zh_2d[:,0], Zh_2d[:,1],
    c=y_vis, s=6, cmap="coolwarm", alpha=0.6
)
plt.title("Hybrid Quantum–Classical Feature Space (t-SNE)")
plt.xlabel("Dim-1")
plt.ylabel("Dim-2")

plt.tight_layout()
plt.savefig(
    "feature_space_tsne_classical_vs_quantum.pdf",
    bbox_inches="tight"
)

plt.show()

print("\n✔ Feature space visualization completed successfully")
